In [ ]:
# ============================================================
# PROJECT : AI Powered Vehicle Valuation System
#
# NOTEBOOK : 06_Model_Training.ipynb
#
# PURPOSE :
# Train a baseline regression model to estimate
# used vehicle market prices.
#
# AUTHOR :
# Srivignesh
#
# MODEL :
# XGBoost Regressor
# ============================================================

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [2]:
# ============================================================
# Load Feature Engineered Dataset
# ============================================================

df = pd.read_csv("../data/vehicle_feature_engineered.csv")

print("Dataset Shape :", df.shape)

df.head()

Dataset Shape : (37006, 15)


,oem,model,variant,myear,fuel,transmission,km,body,owner_type,City,state,listed_price,car_age,premium_brand,price_category
0,maruti,maruti wagon r,lxi cng,2016,cng,manual,69162.0,hatchback,first,lucknow,uttar pradesh,370000.0,10,0,Mid Range
1,maruti,maruti celerio,green vxi,2015,cng,manual,45864.0,hatchback,first,mumbai,maharashtra,365000.0,11,0,Mid Range
2,honda,honda amaze,s plus i-vtec,2015,cng,manual,81506.0,sedan,second,new delhi,delhi,421000.0,11,0,Mid Range
3,maruti,maruti wagon r,lxi cng,2013,cng,manual,115893.0,hatchback,second,new delhi,delhi,240000.0,13,0,Budget
4,maruti,maruti ertiga,vxi cng,2022,cng,manual,18900.0,muv,first,mumbai,maharashtra,1175000.0,4,0,Premium


In [3]:
# ============================================================
# Dataset Validation
# ============================================================

print(df.info())

print()

print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37006 entries, 0 to 37005
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   oem             37006 non-null  object 
 1   model           37006 non-null  object 
 2   variant         37006 non-null  object 
 3   myear           37006 non-null  int64  
 4   fuel            37006 non-null  object 
 5   transmission    37006 non-null  object 
 6   km              37006 non-null  float64
 7   body            37006 non-null  object 
 8   owner_type      37006 non-null  object 
 9   City            37006 non-null  object 
 10  state           37006 non-null  object 
 11  listed_price    37006 non-null  float64
 12  car_age         37006 non-null  int64  
 13  premium_brand   37006 non-null  int64  
 14  price_category  37006 non-null  object 
dtypes: float64(2), int64(3), object(10)
memory usage: 4.2+ MB
None

oem               0
model             0
variant          

In [4]:
# =====================================================
# STEP 3 : Define Input Features and Target Variable
# =====================================================

# Features used for model training
X = df.drop(
    columns=[
        "listed_price",
        "price_category"
    ]
)

# Target variable
y = df["listed_price"]

# Verify dimensions
print("Feature Matrix Shape :", X.shape)

print("Target Shape :", y.shape)

Feature Matrix Shape : (37006, 13)
Target Shape : (37006,)


In [5]:
# =====================================================
# STEP 4 : Validate Feature Data Types
# =====================================================

print(X.dtypes)

oem               object
model             object
variant           object
myear              int64
fuel              object
transmission      object
km               float64
body              object
owner_type        object
City              object
state             object
car_age            int64
premium_brand      int64
dtype: object


In [6]:
# =====================================================
# STEP 5 : Train-Test Split
# =====================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

print("Training Set :", X_train.shape)

print("Testing Set :", X_test.shape)

Training Set : (29604, 13)
Testing Set : (7402, 13)


In [7]:
# =====================================================
# STEP 6.1 : Define Categorical and Numerical Features
# =====================================================

categorical_features = [

    "oem",

    "model",

    "variant",

    "fuel",

    "transmission",

    "body",

    "owner_type",

    "City",

    "state"

]

numerical_features = [

    "km",

    "car_age",

    "premium_brand"

]

print("Categorical Features :", len(categorical_features))

print("Numerical Features :", len(numerical_features))

Categorical Features : 9
Numerical Features : 3


In [8]:
# =====================================================
# STEP 6.2 : Build Preprocessing Pipeline
# =====================================================
# column transformer

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(

    transformers=[

        (

            "categorical",

            OneHotEncoder(

                handle_unknown="ignore"

            ),

            categorical_features

        ),

        (

            "numerical",

            "passthrough",

            numerical_features

        )

    ]

)

In [9]:
# =====================================================
# STEP 7 : Build Baseline Model Pipeline
# =====================================================

from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

baseline_model = Pipeline(

    steps=[

        (

            "preprocessor",

            preprocessor

        ),

        (

            "regressor",

            XGBRegressor(

                objective="reg:squarederror",

                n_estimators=300,

                learning_rate=0.05,

                max_depth=8,

                random_state=42

            )

        )

    ]

)

print(baseline_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['oem', 'model', 'variant',
                                                   'fuel', 'transmission',
                                                   'body', 'owner_type', 'City',
                                                   'state']),
                                                 ('numerical', 'passthrough',
                                                  ['km', 'car_age',
                                                   'premium_brand'])])),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_byle...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
          

In [10]:
# =====================================================
# STEP 8 : Train Baseline Model
# =====================================================

baseline_model.fit(

    X_train,

    y_train

)

print()

print("Baseline Model Training Completed Successfully!")


Baseline Model Training Completed Successfully!


In [11]:
# =====================================================
# STEP 9.1 : Generate Predictions
# =====================================================

# Predict prices for unseen test data

y_pred = baseline_model.predict(X_test)

print("Predictions Generated Successfully!")

print()

print("Total Predictions :", len(y_pred))

Predictions Generated Successfully!

Total Predictions : 7402


In [12]:
# =====================================================
# STEP 9.2 : Calculate Evaluation Metrics
# =====================================================

from sklearn.metrics import (

    mean_absolute_error,

    mean_squared_error,

    r2_score

)

mae = mean_absolute_error(

    y_test,

    y_pred

)

rmse = mean_squared_error(

    y_test,

    y_pred,

    squared=False

)

r2 = r2_score(

    y_test,

    y_pred

)

print("=" * 50)

print("BASELINE MODEL PERFORMANCE")

print("=" * 50)

print()

print(f"Mean Absolute Error : {mae:,.2f}")

print()

print(f"Root Mean Squared Error : {rmse:,.2f}")

print()

print(f"R² Score : {r2:.4f}")

TypeError: got an unexpected keyword argument 'squared'

In [13]:
# =====================================================
# Check Scikit-Learn Version
# =====================================================

import sklearn

print("Scikit-Learn Version :", sklearn.__version__)

Scikit-Learn Version : 1.7.2


In [14]:
# =====================================================
# STEP 9.2 : Calculate Evaluation Metrics
# =====================================================

import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Mean Absolute Error
mae = mean_absolute_error(
    y_test,
    y_pred
)

# Mean Squared Error
mse = mean_squared_error(
    y_test,
    y_pred
)

# Root Mean Squared Error
rmse = np.sqrt(mse)

# R² Score
r2 = r2_score(
    y_test,
    y_pred
)

print("=" * 60)
print("BASELINE MODEL PERFORMANCE")
print("=" * 60)

print(f"\nMean Absolute Error (MAE) : {mae:,.2f}")

print(f"\nRoot Mean Squared Error (RMSE) : {rmse:,.2f}")

print(f"\nR² Score : {r2:.4f}")

BASELINE MODEL PERFORMANCE

Mean Absolute Error (MAE) : 88,889.34

Root Mean Squared Error (RMSE) : 153,777.86

R² Score : 0.9481
